# Core dataset preparation

**Purpose:** Create the reusable publication-level dataset and abstract sidecar for substantive results.

The grain is exactly one screening-eligible publication (`UT`). Screening metadata is joined one-to-one with driver, Threats L0, geography, ecosystem realm, study-design/method, and current taxonomic group fields. List labels are normalized but never exploded. Empty geography and locale lists become `Not Applicable`; empty locale-coordinate JSON lists become `["Not Applicable"]`; and any region, subregion, or country list containing `Unclear` becomes exactly `Unclear`. The three-level geography is audited non-destructively against the IPBES region hierarchy. The original `pred_*` geography fields are retained, while three adjacent `pred_*_audit` fields contain the reviewed labels and use `Unclear - Review needed` for hierarchy-review cases. No direction, year, threat, realm, geography, study, or taxa filter is applied here; every downstream notebook owns its analytical subset.

In [1]:
import sys
from pathlib import Path

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_helpers import results_config
from data_helpers.prep import corpus
from data_helpers.prep import evidence_corpus_prep
from data_helpers.prep import taxa_analysis_prep
from data_helpers._config import get_merged_corpus_config


In [2]:
RESULTS_CONFIG = results_config.load_results_config()
SCREENING_CONFIG = RESULTS_CONFIG["screening_analysis_prep"]
EVIDENCE_CONFIG = RESULTS_CONFIG["biodiversity_evidence_prep"]
TAXA_CONFIG = RESULTS_CONFIG["taxa_analysis_prep"]
CORPUS_CONFIG = get_merged_corpus_config()
IPBES_HIERARCHY = PROJECT_ROOT / "checklists/mappings/ipbes_regions.json"
SCREENING_STORE = evidence_corpus_prep.ScreeningPreparedStore(PROJECT_ROOT / SCREENING_CONFIG["output_directory"])
EVIDENCE_STORE = evidence_corpus_prep.BiodiversityEvidenceStore(PROJECT_ROOT / EVIDENCE_CONFIG["output_directory"])
TAXA_MAPPING = taxa_analysis_prep.load_taxa_mapping(PROJECT_ROOT / TAXA_CONFIG["taxa_group_mapping"])
TAXA_STORE = taxa_analysis_prep.TaxaPreparedStore(PROJECT_ROOT / TAXA_CONFIG["output_directory"])


## 1. Join every coding dimension without changing the eligible UT grain

The eligible screening artifact supplies the left-hand key order. Every configured coding source and the prepared taxa artifact must contain the identical `UT` set and one row per key. The five existing geography and locale fields are standardized in place. Three additional audit-label columns preserve reviewed versions without modifying the source geography fields; assertions protect the publication count, `UT` order, and complete column contract.

In [3]:
eligible_screening, _ = SCREENING_STORE.load_eligible()
merged = corpus.build_merged_corpus_from_eligible_screening(eligible_screening)
taxa_bundle = TAXA_STORE.load(mapping=TAXA_MAPPING)
publications, abstracts = evidence_corpus_prep.build_biodiversity_evidence_corpus(merged, taxa_bundle.articles)
original_ids = publications["id"].copy()
original_uts = publications["UT"].copy()
original_columns = publications.columns.tolist()
normalization_audit = evidence_corpus_prep.standardize_geography_lists(publications)
prepared_geography = publications[["pred_regions", "pred_subregions", "pred_countries"]].copy(deep=True)
hierarchy_audit = evidence_corpus_prep.audit_ipbes_geography_hierarchy(
    publications, mapping_path=IPBES_HIERARCHY
)
audit_label_summary = evidence_corpus_prep.apply_geography_hierarchy_audit_labels(
    publications, hierarchy_audit
)
hierarchy_summary = evidence_corpus_prep.summarize_geography_hierarchy_audit(
    hierarchy_audit
)
assert len(publications) == len(original_uts)
assert publications["id"].equals(original_ids)
assert publications["id"].is_unique
assert publications["UT"].equals(original_uts)
assert publications.columns.tolist() == original_columns
assert publications[["pred_regions", "pred_subregions", "pred_countries"]].equals(prepared_geography)
assert hierarchy_audit["UT"].equals(original_uts)
print(f"Integrated grain: {len(publications):,} rows / {publications['id'].nunique():,} unique ids / {publications['UT'].nunique():,} unique UTs / {len(publications.columns):,} columns.")
display(normalization_audit.style.format({"changed_values": "{:,}"}).hide(axis="index"))
display(hierarchy_summary.style.format({"n_publications": "{:,}"}).hide(axis="index"))
display(audit_label_summary.style.format({"n_publications": "{:,}"}).hide(axis="index"))
display(publications["s2_dir"].value_counts(dropna=False).rename("n_publications").rename_axis("direction").reset_index().style.format({"n_publications": "{:,}"}).hide(axis="index"))


Integrated grain: 605,199 rows / 605,199 unique ids / 605,199 unique UTs / 39 columns.


column,rule,changed_values
locale_coordinates,empty_to_not_applicable,"547,374"
locales,empty_to_not_applicable,"102,641"
pred_countries,all_regions_descendant_to_not_applicable,39
pred_countries,contains_unclear_to_unclear,212
pred_countries,empty_to_not_applicable,"40,495"
pred_regions,empty_to_not_applicable,390
pred_subregions,empty_to_not_applicable,"6,120"


status,primary_reason,n_publications
global,all_regions_descendants_not_applicable,"1,075"
no_geography,all_levels_not_applicable,388
pass_complete,complete_paths_valid,"311,446"
pass_partial,region_subregion_valid_country_non_concrete,"67,411"
pass_region_only,region_valid_descendants_non_concrete,"3,709"
review,all_regions_with_concrete_descendant,"167,698"
review,country_path_missing,"27,220"
review,subregion_parent_missing,"19,076"
review,mixed_special_and_concrete_or_global,"4,594"
review,unknown_country_label,"1,654"


audit_label_action,n_publications
preserved,"384,029"
review_needed,"221,170"


direction,n_publications
negative,"253,081"
mixed,"141,267"
positive,"112,684"
unclear,"98,167"


## 2. Write and verify the dataset files and abstract sidecar

The complete 39-column dataset is written as both `dataset.parquet` and `dataset.xlsx`. Its primary key, `id`, is the stable integer value of the 15-digit WOS `UT` suffix; `UT` remains the external source key. Parquet retains native list fields; the single-sheet Excel copy stores those same lists as JSON text such as `["Americas", "Europe and Central Asia"]`, with no rows or columns omitted. The dataset retains the three original geography labels, three reviewed `pred_*_audit` labels, seven useful taxonomy-rank labels, the publication-level `taxa_record_status`, and two derived text-feature flags, `text_available` and `plastics_mention` (whether title/abstract text mentions plastics or marine-debris terminology; see `evidence_corpus_prep.DEFAULT_PLASTICS_PATTERN`) — computed here, from title and abstract, so that downstream analyses needing them never have to load abstract text themselves. The redundant `authors`, `eligibility`, `taxa_domain_labels`, and `taxa_subkingdom_labels` fields are omitted. Detailed taxonomy summaries are available in `02_taxa_analysis_prep/taxa_publications.parquet`, while item-level matches and full lineage remain in `02_taxa_analysis_prep/taxa_matches.parquet`; neither is duplicated here. `pred_study_design` is stored as a true `list<string>` field in Parquet, and empty lists are standardized to `["Unclear"]`. The publisher `source` field and redundant derived `n_drivers` count are intentionally omitted; analyses can derive `n_drivers` from `driver` when needed. Abstracts remain lossless in a separate three-column `id`/`UT`/`abstract` Parquet, kept out of the main dataset for size and distribution reasons; no active results notebook needs to load it directly. The per-publication geography audit is not written separately: its reviewed labels are in the dataset and its aggregate status and reason counts are in the compact manifest. `wos_categories` is intentionally not retained.

In [4]:
sources = {
    "eligible_screening_artifact": SCREENING_STORE.eligible_path,
    "corpus_config": PROJECT_ROOT / SCREENING_CONFIG["corpus_config"],
    "ipbes_geography_hierarchy": IPBES_HIERARCHY,
    "prepared_taxa_artifact": TAXA_STORE.article_path,
    "prepared_taxa_manifest": TAXA_STORE.manifest_path,
}
sources.update({f"coding_{source['name']}": source["path"] for source in CORPUS_CONFIG["coding_sources"]})
manifest = evidence_corpus_prep.build_biodiversity_manifest(
    publications, geography_hierarchy_audit=hierarchy_audit, abstracts=abstracts, sources=sources, repository_root=PROJECT_ROOT
)
bundle = evidence_corpus_prep.BiodiversityEvidenceBuild(
    publications=publications, abstracts=abstracts, manifest=manifest
)
written = EVIDENCE_STORE.write(bundle)
reloaded = EVIDENCE_STORE.load(columns=["id", "UT", *evidence_corpus_prep.GEOGRAPHY_VALUE_COLUMNS, *evidence_corpus_prep.GEOGRAPHY_AUDIT_LABEL_COLUMNS])
xlsx_contract = EVIDENCE_STORE.validate_xlsx()
reloaded_abstracts = EVIDENCE_STORE.load_abstracts()
assert reloaded.publications["id"].equals(publications["id"])
assert reloaded.publications["UT"].equals(publications["UT"])
assert len(reloaded.publications) == len(publications)
for column in evidence_corpus_prep.GEOGRAPHY_VALUE_COLUMNS:
    assert reloaded.publications[column].equals(publications[column])
for column in evidence_corpus_prep.GEOGRAPHY_AUDIT_LABEL_COLUMNS:
    assert reloaded.publications[column].equals(publications[column])
assert reloaded_abstracts["id"].equals(abstracts["id"])
assert reloaded_abstracts["UT"].equals(abstracts["UT"])
assert xlsx_contract == {"sheet": "dataset", "columns": len(publications.columns)}
print("Verified integrated corpus:")
for path in written:
    print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024**2:.2f} MiB)")
if not EVIDENCE_CONFIG["retain_eligible_build_cache"]:
    removed = SCREENING_STORE.remove_eligible_build_cache()
    if removed:
        print(f"Removed consumed build cache: {SCREENING_STORE.eligible_path.relative_to(PROJECT_ROOT)}")


Verified integrated corpus:
  notebooks/data_processing/outputs/04_dataset/dataset.parquet (67.97 MiB)
  notebooks/data_processing/outputs/04_dataset/dataset.xlsx (188.45 MiB)
  notebooks/data_processing/outputs/04_dataset/dataset_abstracts.parquet (363.79 MiB)
  notebooks/data_processing/outputs/04_dataset/manifest.json (0.00 MiB)
Removed consumed build cache: notebooks/data_processing/outputs/03_screening_analysis_prep/eligible_screening_publications.parquet


## Handoff contract

Results notebooks request only their required columns from `dataset.parquet` through `BiodiversityEvidenceStore`. The stable numeric `id` is the dataset primary key, while `UT` remains available for existing analysis joins and source reconciliation. The prepared geography contract preserves the original list-valued `pred_regions`, `pred_subregions`, and `pred_countries` columns. Their three `pred_*_audit` counterparts contain the reviewed labels: non-review records retain the prepared values, while all three fields become `Unclear - Review needed` for review records. Aggregate hierarchy status and reason counts remain in the manifest; review cases remain available in the original geography columns. World Bank eligibility remains an income-analysis concern rather than a source-geography rule. Analyses requiring item-level taxonomy matches load `02_taxa_analysis_prep/taxa_matches.parquet` by `UT`; analyses needing publication-level taxonomy audit fields use `02_taxa_analysis_prep/taxa_publications.parquet`. `03_realm_composition.ipynb`'s pollution/plastics/exploitation trends read the precomputed `text_available`/`plastics_mention` columns straight off the main dataset rather than joining `dataset_abstracts.parquet` — no active results notebook needs abstract text directly. Other analyses should request only their required main-dataset columns and establish a one-row-per-publication denominator before constructing an exploded attribution table.